# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR² Mental Health Survey Dataset](https://doi.org/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset provides survey data on mental health indicators among residents of Kilifi County, Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
print("Dataset title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("Published:", dataset.metadata.datePublished)
print("Keywords:", dataset.metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets and for each, their fields and columns, **referencing all by their `@id`** fields as per the dataset schema.

In [ ]:
# List all RecordSet @ids
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id):")
for rset in record_sets:
    print(f"  - @id: {rset['@id']}, name: {rset.get('name', 'N/A')}")

# For each RecordSet, show fields and columns @id
print("\nFields and Columns for Each RecordSet:")
for rset in record_sets:
    print(f"\nRecordSet: {rset['@id']}\n  Fields:")
    for field in rset.get('field', []):
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id')}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
        else:
            print(f"    - @id: {field}")
    print("  Columns:")
    for col in rset.get('column', []):
        if isinstance(col, dict):
            print(f"    - @id: {col.get('@id')}, name: {col.get('name', 'N/A')}")
        else:
            print(f"    - @id: {col}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Replace the `@id`s below as appropriate with those shown in the overview above.**

We extract data from the main record set containing participant rows. (Based on the dataset's schema, replace the `record_set_id` with the actual `@id` if needed.)

In [ ]:
# Example: Extract data for each record set
# You may need to adjust the record set @ids to match those listed above.

record_set_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    # Only create DataFrame if records exist and are dict-like
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Pick one RecordSet as main for demonstration (replace with actual @id if known, here for demo)
main_record_set_id = record_set_ids[0]

print("Columns in main DataFrame:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: count records, filter by a numeric field, normalize, and group by demographics.

Pick a numeric field (e.g., a score column such as `gad7_score` or `phq9_score`) and a group field (e.g., `village` or `gender`).

Replace these field `@id`s with the actual ones from the record set's fields overview.

In [ ]:
# Identify numeric and group fields by their column names or field @id
# Replace 'gad7_score' and 'village' with actual field names as needed
df = dataframes[main_record_set_id]

# For demonstration, infer some likely fields:
numeric_field = None
for col in df.columns:
    if "gad7" in col or "score" in col or "phq9" in col or "psq" in col:
        numeric_field = col
        print(f"Selected numeric field: {numeric_field}")
        break
if numeric_field is None:
    numeric_field = df.select_dtypes("number").columns[0]
    print(f"Defaulting to numeric field: {numeric_field}")

# Filter records above a threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)} records where {numeric_field} > {threshold}")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"\nFirst few normalized {numeric_field} values:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field (e.g., village, gender)
group_field = None
for col in df.columns:
    if "village" in col.lower() or "gender" in col.lower() or "education" in col.lower():
        group_field = col
        print(f"Grouping by: {group_field}")
        break
if group_field is not None:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped means of {numeric_field} by {group_field}:")
    print(grouped.head())


## 5. Visualization
Visualize data distributions and relationships between fields. Here, we show a histogram of the selected numeric field, and a boxplot grouped by a demographic field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field, if available
if group_field is not None:
    plt.figure(figsize=(8,4))
    order = df[group_field].value_counts().index[:10] # Only top categories for clarity
    sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
This notebook demonstrated FAIR, programmatic exploration of the Kilifi County Mental Health Survey using the `mlcroissant` library, including:

- Loading dataset metadata and records using the Croissant schema
- Reviewing and extracting schema structure (record sets, fields, and columns by their `@id`)
- Loading participant data into pandas DataFrames for interactive analysis
- Filtering, normalizing, and grouping data by demographic fields
- Visualizing survey score distributions and group differences

This structured workflow enables reproducible, AI-ready analysis in accordance with modern dataset documentation standards. For more advanced exploration, see the [FAIR² documentation](https://doi.org/10.71728/senscience.vcs2-05nj) and the [mlcroissant examples](https://github.com/mlcommons/croissant/tree/main/examples).
